# Admission-time prediction of myocardial infarction relapse

This notebook predicts myocardial infarction relapse (`REC_IM`) from information available at hospital admission.

## Reproducible environment
1. `uv venv .venv --python 3.12.2`
2. `uv sync`
3. `uv run jupyter nbconvert --to notebook --execute mi_complication_project.ipynb --inplace --ExecutePreprocessor.timeout=1200`


In [1]:
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    silhouette_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)
sns.set_theme(style='whitegrid')


In [2]:
DATA_PATH = Path('data') / 'MI.data'
REPORT_DIR = Path('report')
FIGURES_DIR = REPORT_DIR / 'figures'
TABLES_DIR = REPORT_DIR / 'tables'
for output_dir in [REPORT_DIR, FIGURES_DIR, TABLES_DIR]:
    output_dir.mkdir(parents=True, exist_ok=True)

COLUMN_NAMES = [
    'ID', 'AGE', 'SEX', 'INF_ANAM', 'STENOK_AN', 'FK_STENOK', 'IBS_POST', 'IBS_NASL', 'GB', 'SIM_GIPERT',
    'DLIT_AG', 'ZSN_A', 'nr_11', 'nr_01', 'nr_02', 'nr_03', 'nr_04', 'nr_07', 'nr_08', 'np_01',
    'np_04', 'np_05', 'np_07', 'np_08', 'np_09', 'np_10', 'endocr_01', 'endocr_02', 'endocr_03', 'zab_leg_01',
    'zab_leg_02', 'zab_leg_03', 'zab_leg_04', 'zab_leg_06', 'S_AD_KBRIG', 'D_AD_KBRIG', 'S_AD_ORIT', 'D_AD_ORIT', 'O_L_POST', 'K_SH_POST',
    'MP_TP_POST', 'SVT_POST', 'GT_POST', 'FIB_G_POST', 'ant_im', 'lat_im', 'inf_im', 'post_im', 'IM_PG_P', 'ritm_ecg_p_01',
    'ritm_ecg_p_02', 'ritm_ecg_p_04', 'ritm_ecg_p_06', 'ritm_ecg_p_07', 'ritm_ecg_p_08', 'n_r_ecg_p_01', 'n_r_ecg_p_02', 'n_r_ecg_p_03', 'n_r_ecg_p_04', 'n_r_ecg_p_05',
    'n_r_ecg_p_06', 'n_r_ecg_p_08', 'n_r_ecg_p_09', 'n_r_ecg_p_10', 'n_p_ecg_p_01', 'n_p_ecg_p_03', 'n_p_ecg_p_04', 'n_p_ecg_p_05', 'n_p_ecg_p_06', 'n_p_ecg_p_07',
    'n_p_ecg_p_08', 'n_p_ecg_p_09', 'n_p_ecg_p_10', 'n_p_ecg_p_11', 'n_p_ecg_p_12', 'fibr_ter_01', 'fibr_ter_02', 'fibr_ter_03', 'fibr_ter_05', 'fibr_ter_06',
    'fibr_ter_07', 'fibr_ter_08', 'GIPO_K', 'K_BLOOD', 'GIPER_NA', 'NA_BLOOD', 'ALT_BLOOD', 'AST_BLOOD', 'KFK_BLOOD', 'L_BLOOD',
    'ROE', 'TIME_B_S', 'R_AB_1_n', 'R_AB_2_n', 'R_AB_3_n', 'NA_KB', 'NOT_NA_KB', 'LID_KB', 'NITR_S', 'NA_R_1_n',
    'NA_R_2_n', 'NA_R_3_n', 'NOT_NA_1_n', 'NOT_NA_2_n', 'NOT_NA_3_n', 'LID_S_n', 'B_BLOK_S_n', 'ANT_CA_S_n', 'GEPAR_S_n', 'ASP_S_n',
    'TIKL_S_n', 'TRENT_S_n', 'FIBR_PREDS', 'PREDS_TAH', 'JELUD_TAH', 'FIBR_JELUD', 'A_V_BLOK', 'OTEK_LANC', 'RAZRIV', 'DRESSLER',
    'ZSN', 'REC_IM', 'P_IM_STEN', 'LET_IS'
]

TARGET_COLUMN = 'REC_IM'
TIMEPOINT = 'admission'
ID_COLUMN = 'ID'
RANDOM_STATE = 42
HIGH_MISSINGNESS_THRESHOLD = 0.40
SIGMA_CLIPPING_N = 3
DEFAULT_MODEL_METRIC = 'balanced_accuracy'
ADMISSION_FORBIDDEN_COLUMN_NUMBERS = [93, 94, 95, 100, 101, 102, 103, 104, 105]
ADMISSION_FORBIDDEN_COLUMNS = [
    'R_AB_1_n', 'R_AB_2_n', 'R_AB_3_n', 'NA_R_1_n', 'NA_R_2_n', 'NA_R_3_n',
    'NOT_NA_1_n', 'NOT_NA_2_n', 'NOT_NA_3_n',
]
OUTPUT_COLUMNS = COLUMN_NAMES[112:]
NON_TARGET_OUTPUT_COLUMNS = [column for column in OUTPUT_COLUMNS if column != TARGET_COLUMN]
EDA_FOCUS_COLUMNS = ['AGE', 'TIME_B_S', 'K_BLOOD', 'ROE']
KMEANS_K_RANGE = range(2, 9)

CLASS_BALANCE_FIGURE = FIGURES_DIR / 'class_balance.svg'
MISSINGNESS_FIGURE = FIGURES_DIR / 'missingness_top15.svg'
DISTRIBUTIONS_FIGURE = FIGURES_DIR / 'key_distributions.svg'
CORRELATION_FIGURE = FIGURES_DIR / 'correlation_heatmap.svg'
ROC_CURVES_FIGURE = FIGURES_DIR / 'roc_curves.svg'
THRESHOLD_TRADEOFFS_FIGURE = FIGURES_DIR / 'threshold_tradeoffs.svg'
TREE_PRUNING_CURVE_FIGURE = FIGURES_DIR / 'tree_pruning_curve.svg'
SILHOUETTE_FIGURE = FIGURES_DIR / 'silhouette_scores.svg'
CLUSTER_PCA_FIGURE = FIGURES_DIR / 'cluster_pca.svg'

EXCLUSION_AUDIT_TABLE = TABLES_DIR / 'admission_exclusion_audit.csv'
CLASS_BALANCE_TABLE = TABLES_DIR / 'class_balance.csv'
MISSINGNESS_SUMMARY_TABLE = TABLES_DIR / 'missingness_summary.csv'
FEATURE_HANDLING_TABLE = TABLES_DIR / 'feature_handling.csv'
TOP_CORRELATIONS_TABLE = TABLES_DIR / 'top_correlations.csv'
SIGMA_CLIPPING_TABLE = TABLES_DIR / 'sigma_clipping_summary.csv'
CLUSTERING_RESULTS_TABLE = TABLES_DIR / 'clustering_results.csv'
CV_RESULTS_SUMMARY_TABLE = TABLES_DIR / 'cv_results_summary.csv'
CV_FOLD_RESULTS_TABLE = TABLES_DIR / 'cv_fold_results.csv'
CV_FOLD_AUDIT_TABLE = TABLES_DIR / 'cv_fold_preprocessing_audit.csv'
BASELINE_VS_WEIGHTED_TABLE = TABLES_DIR / 'baseline_vs_weighted.csv'
THRESHOLD_SELECTION_TABLE = TABLES_DIR / 'threshold_selection.csv'
FINAL_MODEL_METRICS_TABLE = TABLES_DIR / 'final_model_metrics.csv'
FINAL_CONFUSION_MATRICES_TABLE = TABLES_DIR / 'final_confusion_matrices.csv'
EVALUATION_PROTOCOL_TABLE = TABLES_DIR / 'evaluation_protocol.csv'


In [3]:
def save_figure(fig: plt.Figure, svg_path: Path) -> None:
    """Save matching vector files and normalize SVG line endings."""
    fig.savefig(svg_path, bbox_inches='tight')
    svg_text = svg_path.read_text(encoding='utf-8')
    normalized_svg = '\n'.join(line.rstrip() for line in svg_text.splitlines()) + '\n'
    svg_path.write_text(normalized_svg, encoding='utf-8', newline='\n')
    fig.savefig(svg_path.with_suffix('.pdf'), bbox_inches='tight')
    plt.close(fig)


In [4]:
def fit_preprocessing(feature_df: pd.DataFrame, stage: str) -> tuple[dict, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Learn missingness, imputation, and clipping parameters from one training partition."""
    missingness = pd.DataFrame({
        'feature': feature_df.columns,
        'missing_count': feature_df.isna().sum().values,
        'missing_share': feature_df.isna().mean().values,
    })
    missingness['missing_pct'] = missingness['missing_share'] * 100
    missingness['fitted_on'] = stage
    missingness = missingness.sort_values(['missing_share', 'feature'], ascending=[False, True]).reset_index(drop=True)

    dropped = missingness.loc[missingness['missing_share'] > HIGH_MISSINGNESS_THRESHOLD, 'feature'].tolist()
    retained = [column for column in feature_df.columns if column not in dropped]
    fit_values = feature_df[retained].copy()
    fill_values = {}
    strategies = {}
    continuous_columns = []
    handling_rows = []

    for column in retained:
        unique_values = fit_values[column].dropna().nunique()
        if unique_values <= 10:
            strategy = 'mode'
            fill_value = fit_values[column].mode(dropna=True).iloc[0]
        else:
            strategy = 'median'
            fill_value = fit_values[column].median()
            continuous_columns.append(column)
        fill_values[column] = float(fill_value)
        strategies[column] = strategy
        handling_rows.append({
            'feature': column,
            'strategy': strategy,
            'fill_value': float(fill_value),
            'training_missing_count': int(feature_df[column].isna().sum()),
            'fitted_on': stage,
        })
        fit_values[column] = fit_values[column].fillna(fill_value)

    clip_bounds = {}
    clip_rows = []
    for column in continuous_columns:
        mean = float(fit_values[column].mean())
        std = float(fit_values[column].std())
        lower = mean - SIGMA_CLIPPING_N * std
        upper = mean + SIGMA_CLIPPING_N * std
        n_below = int((fit_values[column] < lower).sum())
        n_above = int((fit_values[column] > upper).sum())
        clip_bounds[column] = (lower, upper)
        clip_rows.append({
            'feature': column,
            'mean': mean,
            'std': std,
            'lower_bound': lower,
            'upper_bound': upper,
            'n_clipped_below_training': n_below,
            'n_clipped_above_training': n_above,
            'n_clipped_training': n_below + n_above,
            'pct_clipped_training': (n_below + n_above) / len(feature_df) * 100,
            'fitted_on': stage,
        })

    parameters = {
        'stage': stage,
        'fit_indices': tuple(feature_df.index.tolist()),
        'retained_columns': retained,
        'dropped_columns': dropped,
        'fill_values': fill_values,
        'strategies': strategies,
        'clip_bounds': clip_bounds,
    }
    return parameters, missingness, pd.DataFrame(handling_rows), pd.DataFrame(clip_rows)


def apply_preprocessing(feature_df: pd.DataFrame, parameters: dict) -> pd.DataFrame:
    """Apply training-derived preprocessing without recalculating any parameter."""
    transformed = feature_df[parameters['retained_columns']].copy()
    for column, fill_value in parameters['fill_values'].items():
        transformed[column] = transformed[column].fillna(fill_value)
    for column, (lower, upper) in parameters['clip_bounds'].items():
        transformed[column] = transformed[column].clip(lower=lower, upper=upper)
    assert list(transformed.columns) == parameters['retained_columns']
    assert int(transformed.isna().sum().sum()) == 0
    return transformed


def run_kmeans_clustering(feature_df: pd.DataFrame, k_range: range = KMEANS_K_RANGE):
    """Scale the descriptive matrix and select K by silhouette score."""
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(feature_df)
    result_rows = []
    for k in k_range:
        labels = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit_predict(X_scaled)
        score = float(silhouette_score(X_scaled, labels))
        sizes = pd.Series(labels).value_counts().sort_index().to_dict()
        result_rows.append({'k': int(k), 'silhouette_score': score, 'cluster_sizes': str(sizes)})
    results = pd.DataFrame(result_rows)
    best_k = int(results.loc[results['silhouette_score'].idxmax(), 'k'])
    best_labels = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10).fit_predict(X_scaled)
    return best_k, best_labels, results, X_scaled


def plot_clustering(X_scaled, labels, results, target, best_k) -> None:
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(results['k'], results['silhouette_score'], marker='o', color='#4c78a8')
    best_score = float(results.loc[results['k'] == best_k, 'silhouette_score'].iloc[0])
    ax.scatter([best_k], [best_score], color='#f58518', s=100, zorder=3, label=f'best K={best_k}')
    ax.set(xlabel='Number of clusters (K)', ylabel='Silhouette score', title='K-means silhouette score by number of clusters')
    ax.legend()
    fig.tight_layout()
    save_figure(fig, SILHOUETTE_FIGURE)

    # PCA is used only to display the fitted clusters in two dimensions.
    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    coordinates = pca.fit_transform(X_scaled)
    palette = matplotlib.colormaps.get_cmap('tab10').resampled(best_k)
    fig, ax = plt.subplots(figsize=(8, 6))
    for cluster_id in range(best_k):
        mask = labels == cluster_id
        relapse_pct = float(target.to_numpy()[mask].mean() * 100)
        ax.scatter(
            coordinates[mask, 0], coordinates[mask, 1], color=palette(cluster_id), alpha=0.5, s=15,
            label=f'Cluster {cluster_id} (n={mask.sum()}, {relapse_pct:.1f}% REC_IM=1)',
        )
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}% variance)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}% variance)')
    ax.set_title(f'K-means clusters (K={best_k}) — PCA display only')
    ax.legend(fontsize=8)
    fig.tight_layout()
    save_figure(fig, CLUSTER_PCA_FIGURE)


In [5]:
def load_raw_dataset(data_path: Path = DATA_PATH) -> pd.DataFrame:
    return pd.read_csv(data_path, header=None, names=COLUMN_NAMES, na_values='?')


def build_admission_column_audit() -> pd.DataFrame:
    rows = []
    for number, name in enumerate(COLUMN_NAMES, start=1):
        if name == ID_COLUMN:
            rows.append([number, name, 'identifier', 'excluded', 'Reference-only identifier'])
        elif name == TARGET_COLUMN:
            rows.append([number, name, 'target', 'target', 'Prediction target'])
        elif number in ADMISSION_FORBIDDEN_COLUMN_NUMBERS:
            rows.append([number, name, 'input', 'excluded', 'Unavailable at admission'])
        elif name in NON_TARGET_OUTPUT_COLUMNS:
            rows.append([number, name, 'output', 'excluded', 'Non-target outcome'])
        else:
            rows.append([number, name, 'input', 'included', 'Admission-time candidate predictor'])
    return pd.DataFrame(rows, columns=['column_number', 'column_name', 'group', 'status', 'reason'])


def build_modeling_dataframe(raw_df: pd.DataFrame):
    audit = build_admission_column_audit()
    dropped = audit.loc[audit['status'] == 'excluded', 'column_name'].tolist()
    model_df = raw_df.drop(columns=dropped)
    feature_columns = [column for column in model_df.columns if column != TARGET_COLUMN]
    assert ID_COLUMN not in model_df.columns
    assert not set(ADMISSION_FORBIDDEN_COLUMNS).intersection(feature_columns)
    assert not set(NON_TARGET_OUTPUT_COLUMNS).intersection(feature_columns)
    return model_df, audit, feature_columns


def compute_top_target_correlations(feature_df: pd.DataFrame, target: pd.Series, top_n: int = 10) -> pd.DataFrame:
    correlations = pd.concat([feature_df, target], axis=1).corr(numeric_only=True)[TARGET_COLUMN].drop(TARGET_COLUMN)
    selected = correlations.abs().sort_values(ascending=False).head(top_n).index
    return pd.DataFrame({
        'feature': selected,
        'correlation_with_target': correlations.loc[selected].values,
        'absolute_correlation': correlations.loc[selected].abs().values,
    })


def plot_and_export_eda(feature_df, target, missingness_summary, top_correlations) -> None:
    class_balance = target.value_counts().sort_index().rename_axis(TARGET_COLUMN).reset_index(name='count')
    class_balance['share'] = class_balance['count'] / len(target)
    class_balance.to_csv(CLASS_BALANCE_TABLE, index=False)
    missingness_summary.to_csv(MISSINGNESS_SUMMARY_TABLE, index=False)
    top_correlations.to_csv(TOP_CORRELATIONS_TABLE, index=False)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(class_balance[TARGET_COLUMN].astype(str), class_balance['count'], color=['#4c78a8', '#f58518'])
    ax.set(xlabel='REC_IM', ylabel='Count', title='REC_IM class balance in the fit partition')
    fig.tight_layout()
    save_figure(fig, CLASS_BALANCE_FIGURE)

    fig, ax = plt.subplots(figsize=(10, 5))
    top_missing = missingness_summary.head(15).sort_values('missing_share')
    ax.barh(top_missing['feature'], top_missing['missing_pct'], color='#72b7b2')
    ax.axvline(HIGH_MISSINGNESS_THRESHOLD * 100, color='black', linestyle='--', linewidth=1, label='40% threshold')
    ax.set(xlabel='Missing percentage in fit data', ylabel='Feature', title='Top 15 fit-partition missingness percentages')
    ax.legend()
    fig.tight_layout()
    save_figure(fig, MISSINGNESS_FIGURE)

    distribution_df = pd.concat([feature_df[EDA_FOCUS_COLUMNS], target], axis=1)
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for axis, column in zip(axes.flatten(), EDA_FOCUS_COLUMNS):
        sns.boxplot(data=distribution_df, x=TARGET_COLUMN, y=column, ax=axis)
        axis.set_title(f'{column} by {TARGET_COLUMN}')
    fig.tight_layout()
    save_figure(fig, DISTRIBUTIONS_FIGURE)

    correlation_features = top_correlations['feature'].tolist()
    correlation_df = pd.concat([feature_df[correlation_features], target], axis=1).corr(numeric_only=True)
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(correlation_df, cmap='coolwarm', center=0, ax=ax)
    ax.set_title('Correlations among the strongest target-linked features')
    fig.tight_layout()
    save_figure(fig, CORRELATION_FIGURE)


def build_model_specs() -> dict:
    return {
        'LogisticRegression': {
            'factory': LogisticRegression,
            'base_params': {'max_iter': 2000, 'solver': 'liblinear', 'random_state': RANDOM_STATE},
            'param_grid': {'C': [0.1, 1.0, 5.0], 'class_weight': [None, 'balanced']},
            'needs_scaling': True,
            'default_threshold': 0.5,
        },
        'DecisionTreeClassifier': {
            'factory': DecisionTreeClassifier,
            'base_params': {'criterion': 'entropy', 'random_state': RANDOM_STATE},
            'param_grid': {
                'max_depth': [3, 5, None], 'min_samples_leaf': [1, 5, 10],
                'class_weight': [None, 'balanced'], 'ccp_alpha': [0.0, 0.0005, 0.001, 0.005, 0.01],
            },
            'needs_scaling': False,
            'default_threshold': 0.5,
        },
        'SVC': {
            'factory': SVC,
            'base_params': {'gamma': 'scale'},
            'param_grid': {'C': [0.1, 1.0, 5.0], 'kernel': ['linear', 'rbf'], 'class_weight': [None, 'balanced']},
            'needs_scaling': True,
            'default_threshold': 0.0,
        },
    }


def expand_param_grid(param_grid: dict) -> list[dict]:
    """Create a deterministic Cartesian product without additional model-selection utilities."""
    combinations = [{}]
    for name, values in param_grid.items():
        combinations = [{**existing, name: value} for existing in combinations for value in values]
    return combinations


def instantiate_model(spec: dict, extra_params: dict | None = None):
    parameters = spec['base_params'].copy()
    if extra_params:
        parameters.update(extra_params)
    return spec['factory'](**parameters)


def class_weight_label(value) -> str:
    if value == 'balanced':
        return 'balanced'
    if value is None or pd.isna(value):
        return 'None'
    return str(value)


def format_best_params(params: dict) -> str:
    return '; '.join(f'{key}={params[key]}' for key in sorted(params))


def get_model_scores(model, X) -> np.ndarray:
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, 'decision_function'):
        return model.decision_function(X)
    raise ValueError('Model does not expose continuous classification scores.')


def build_stratified_folds(target: pd.Series, n_splits: int = 5) -> list[tuple[np.ndarray, np.ndarray]]:
    """Build deterministic stratified fold positions using NumPy only."""
    target_values = target.to_numpy()
    fold_assignment = np.full(len(target_values), -1, dtype=int)
    random_generator = np.random.default_rng(RANDOM_STATE)
    for class_value in np.sort(np.unique(target_values)):
        class_positions = np.flatnonzero(target_values == class_value)
        shuffled_positions = random_generator.permutation(class_positions)
        for order, position in enumerate(shuffled_positions):
            fold_assignment[position] = order % n_splits
    assert np.all(fold_assignment >= 0)

    all_positions = np.arange(len(target_values))
    folds = []
    validation_positions_seen = []
    for fold_number in range(n_splits):
        validation_positions = all_positions[fold_assignment == fold_number]
        training_positions = all_positions[fold_assignment != fold_number]
        assert set(training_positions).isdisjoint(set(validation_positions))
        assert len(np.unique(target_values[validation_positions])) == len(np.unique(target_values))
        folds.append((training_positions, validation_positions))
        validation_positions_seen.extend(validation_positions.tolist())
    assert sorted(validation_positions_seen) == all_positions.tolist()
    return folds


def fit_optional_scaler(X_train, X_other, needs_scaling: bool):
    """Fit StandardScaler on training rows only and transform another partition."""
    if not needs_scaling:
        return None, X_train.to_numpy(), X_other.to_numpy()
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    assert int(scaler.n_samples_seen_) == len(X_train)
    return scaler, X_train_scaled, scaler.transform(X_other)


def run_manual_cross_validation(
    X_fit_raw: pd.DataFrame,
    y_fit: pd.Series,
    reserved_indices: set,
    n_splits: int = 5,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, str, dict]:
    """Evaluate every model combination with fold-local preprocessing and scaling."""
    assert set(X_fit_raw.index).isdisjoint(reserved_indices)
    folds = build_stratified_folds(y_fit, n_splits=n_splits)
    model_specs = build_model_specs()
    result_rows = []
    audit_rows = []

    for model_name, spec in model_specs.items():
        for model_params in expand_param_grid(spec['param_grid']):
            parameter_signature = format_best_params(model_params)
            fold_scores = []
            for fold_number, (training_positions, validation_positions) in enumerate(folds, start=1):
                fold_train_raw = X_fit_raw.iloc[training_positions]
                fold_validation_raw = X_fit_raw.iloc[validation_positions]
                fold_y_train = y_fit.iloc[training_positions]
                fold_y_validation = y_fit.iloc[validation_positions]
                fold_train_indices = set(fold_train_raw.index)
                fold_validation_indices = set(fold_validation_raw.index)

                assert fold_train_indices.isdisjoint(fold_validation_indices)
                assert fold_train_indices.isdisjoint(reserved_indices)
                fold_parameters, _, _, _ = fit_preprocessing(
                    fold_train_raw,
                    f'manual CV fold {fold_number} train: {model_name}; {parameter_signature}',
                )
                assert set(fold_parameters['fit_indices']) == fold_train_indices
                assert set(fold_parameters['fit_indices']).isdisjoint(fold_validation_indices | reserved_indices)

                X_fold_train = apply_preprocessing(fold_train_raw, fold_parameters)
                X_fold_validation = apply_preprocessing(fold_validation_raw, fold_parameters)
                assert list(X_fold_train.columns) == list(X_fold_validation.columns)
                scaler, X_fold_train_model, X_fold_validation_model = fit_optional_scaler(
                    X_fold_train, X_fold_validation, spec['needs_scaling']
                )
                model = instantiate_model(spec, model_params)
                model.fit(X_fold_train_model, fold_y_train)
                fold_predictions = model.predict(X_fold_validation_model)
                fold_score = float(balanced_accuracy_score(fold_y_validation, fold_predictions))
                fold_scores.append(fold_score)
                audit_rows.append({
                    'model': model_name,
                    'parameter_signature': parameter_signature,
                    'fold': fold_number,
                    'preprocessor_fit_rows': len(fold_train_raw),
                    'fold_validation_rows': len(fold_validation_raw),
                    'scaler_fit_rows': len(fold_train_raw) if scaler is not None else 0,
                    'feature_count': X_fold_train.shape[1],
                    'train_validation_disjoint': fold_train_indices.isdisjoint(fold_validation_indices),
                    'reserved_indices_excluded': fold_train_indices.isdisjoint(reserved_indices),
                    'schema_match': list(X_fold_train.columns) == list(X_fold_validation.columns),
                })

            result_row = {
                'model': model_name,
                'parameter_signature': parameter_signature,
                'mean_test_score': float(np.mean(fold_scores)),
                'std_test_score': float(np.std(fold_scores, ddof=0)),
            }
            for fold_number, fold_score in enumerate(fold_scores, start=1):
                result_row[f'fold_{fold_number}_balanced_accuracy'] = fold_score
            for parameter_name, parameter_value in model_params.items():
                result_row[f'param_{parameter_name}'] = parameter_value
            result_rows.append(result_row)

    all_results = pd.DataFrame(result_rows)
    sorted_results = all_results.sort_values(
        ['model', 'mean_test_score', 'std_test_score', 'parameter_signature'],
        ascending=[True, False, True, True],
    )
    # Keep the complete winning row; groupby.first() would skip valid None/NaN
    # parameters and could combine values from different configurations.
    family_best = sorted_results.drop_duplicates(subset='model', keep='first').copy()
    cv_summary = family_best.sort_values(
        ['mean_test_score', 'std_test_score', 'model'], ascending=[False, True, True]
    ).reset_index(drop=True)
    cv_summary.insert(0, 'cv_rank', np.arange(1, len(cv_summary) + 1))
    cv_summary = cv_summary.rename(columns={
        'mean_test_score': 'best_cv_balanced_accuracy',
        'std_test_score': 'cv_score_std',
        'parameter_signature': 'best_params',
    })
    cv_summary.insert(2, 'selection_criterion', 'highest mean fold-local CV balanced accuracy; lower standard deviation and parameter signature break ties')
    cv_summary['needs_scaling'] = cv_summary['model'].map({name: spec['needs_scaling'] for name, spec in model_specs.items()})
    cv_summary['selected_class_weight'] = cv_summary['param_class_weight'].map(class_weight_label)
    cv_summary['default_threshold'] = cv_summary['model'].map({name: spec['default_threshold'] for name, spec in model_specs.items()})

    selected_model_name = str(cv_summary.iloc[0]['model'])
    selected_signature = str(cv_summary.iloc[0]['best_params'])
    selected_params = next(
        params for params in expand_param_grid(model_specs[selected_model_name]['param_grid'])
        if format_best_params(params) == selected_signature
    )
    class_weight_rows = [summarize_class_weight_effect(name, all_results.loc[all_results['model'] == name]) for name in model_specs]
    baseline_vs_weighted = pd.DataFrame(class_weight_rows).sort_values('model').reset_index(drop=True)
    fold_audit = pd.DataFrame(audit_rows)
    assert len(fold_audit) == len(all_results) * n_splits
    assert fold_audit[['train_validation_disjoint', 'reserved_indices_excluded', 'schema_match']].all().all()
    return cv_summary, all_results, fold_audit, baseline_vs_weighted, selected_model_name, selected_params


def select_threshold(y_true, scores, default_threshold):
    scores = np.asarray(scores, dtype=float)
    thresholds = np.unique(np.append(np.linspace(scores.min(), scores.max(), 41), default_threshold))
    rows = []
    for threshold in np.sort(thresholds):
        predictions = (scores >= threshold).astype(int)
        rows.append({
            'threshold': float(threshold),
            'accuracy': accuracy_score(y_true, predictions),
            'balanced_accuracy': balanced_accuracy_score(y_true, predictions),
            'recall': recall_score(y_true, predictions, zero_division=0),
            'precision': precision_score(y_true, predictions, zero_division=0),
            'f1': f1_score(y_true, predictions, zero_division=0),
            'positive_prediction_rate': float(predictions.mean()),
            'distance_from_default': abs(float(threshold) - float(default_threshold)),
            'is_default': bool(np.isclose(threshold, default_threshold)),
        })
    table = pd.DataFrame(rows)
    selected_index = table.sort_values(
        ['balanced_accuracy', 'recall', 'precision', 'f1', 'distance_from_default'],
        ascending=[False, False, False, False, True],
    ).index[0]
    table['selected'] = False
    table.loc[selected_index, 'selected'] = True
    return float(table.loc[selected_index, 'threshold']), table.sort_values('threshold').reset_index(drop=True)


def evaluate_predictions(model_name, y_true, scores, threshold, best_params) -> dict:
    predictions = (np.asarray(scores) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    return {
        'test_evaluation_order': 1,
        'model': model_name,
        'class_weight': class_weight_label(best_params.get('class_weight')),
        'selected_threshold': float(threshold),
        'best_params': format_best_params(best_params),
        'accuracy': accuracy_score(y_true, predictions),
        'balanced_accuracy': balanced_accuracy_score(y_true, predictions),
        'recall': recall_score(y_true, predictions, zero_division=0),
        'precision': precision_score(y_true, predictions, zero_division=0),
        'f1': f1_score(y_true, predictions, zero_division=0),
        'roc_auc': roc_auc_score(y_true, scores),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }


def compute_roc_curve_points(y_true, scores) -> pd.DataFrame:
    rows = []
    for threshold in np.r_[np.inf, np.sort(np.unique(np.asarray(scores, dtype=float)))[::-1]]:
        tn, fp, fn, tp = confusion_matrix(y_true, scores >= threshold, labels=[0, 1]).ravel()
        rows.append({'fpr': fp / (fp + tn) if fp + tn else 0.0, 'tpr': tp / (tp + fn) if tp + fn else 0.0})
    return pd.DataFrame(rows)


def summarize_class_weight_effect(model_name: str, cv_results: pd.DataFrame) -> dict:
    labels = cv_results['param_class_weight'].map(class_weight_label)
    baseline_best = cv_results.loc[labels == 'None'].sort_values('mean_test_score', ascending=False).iloc[0]
    weighted_best = cv_results.loc[labels == 'balanced'].sort_values('mean_test_score', ascending=False).iloc[0]
    preferred = 'balanced' if weighted_best['mean_test_score'] > baseline_best['mean_test_score'] else 'None'
    return {
        'model': model_name,
        'baseline_class_weight': 'None',
        'baseline_cv_balanced_accuracy': float(baseline_best['mean_test_score']),
        'weighted_class_weight': 'balanced',
        'weighted_cv_balanced_accuracy': float(weighted_best['mean_test_score']),
        'weighted_minus_baseline': float(weighted_best['mean_test_score'] - baseline_best['mean_test_score']),
        'preferred_class_weight_from_cv': preferred,
    }


def plot_roc_curve(model_name, roc_auc, curve) -> None:
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(curve['fpr'], curve['tpr'], label=f'{model_name} (AUC={roc_auc:.3f})')
    ax.plot([0, 1], [0, 1], linestyle='--', color='black', linewidth=1)
    ax.set(xlabel='False positive rate', ylabel='True positive rate', title='Final model ROC curve on the held-out test set')
    ax.legend()
    fig.tight_layout()
    save_figure(fig, ROC_CURVES_FIGURE)


def plot_threshold_tradeoffs(threshold_table, model_name) -> None:
    fig, ax = plt.subplots(figsize=(8, 5))
    for metric, color in [('balanced_accuracy', '#4c78a8'), ('recall', '#f58518'), ('precision', '#54a24b'), ('f1', '#e45756')]:
        ax.plot(threshold_table['threshold'], threshold_table[metric], label=metric, color=color)
    selected = threshold_table.loc[threshold_table['selected']].iloc[0]
    ax.axvline(selected['threshold'], color='black', linestyle=':', linewidth=1.5, label='selected threshold')
    ax.scatter(selected['threshold'], selected['balanced_accuracy'], color='#4c78a8', s=60, zorder=3)
    ax.set(xlabel='Validation threshold', ylabel='Metric value', title=f'Validation threshold trade-offs for {model_name}')
    ax.legend()
    fig.tight_layout()
    save_figure(fig, THRESHOLD_TRADEOFFS_FIGURE)


def plot_tree_pruning_curve(tree_cv_results) -> None:
    grouped = tree_cv_results.groupby('param_ccp_alpha', dropna=False)['mean_test_score'].max().reset_index()
    grouped = grouped.rename(columns={'param_ccp_alpha': 'ccp_alpha', 'mean_test_score': 'best_cv_balanced_accuracy'}).sort_values('ccp_alpha')
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(grouped['ccp_alpha'], grouped['best_cv_balanced_accuracy'], marker='o', color='#72b7b2')
    ax.set(xlabel='ccp_alpha', ylabel='Best CV balanced accuracy', title='Decision-tree pruning curve from manual fold-local CV')
    fig.tight_layout()
    save_figure(fig, TREE_PRUNING_CURVE_FIGURE)


def train_select_and_test(
    X_fit_raw,
    X_fit,
    final_parameters,
    X_validation_raw,
    X_test_raw,
    y_fit,
    y_validation,
    y_test,
    reserved_indices,
):
    """Select by fold-local CV, tune one threshold, and evaluate the same fitted model once."""
    (
        cv_summary,
        cv_fold_results,
        cv_fold_audit,
        baseline_vs_weighted,
        selected_model_name,
        selected_params,
    ) = run_manual_cross_validation(X_fit_raw, y_fit, reserved_indices)
    model_specs = build_model_specs()
    selected_spec = model_specs[selected_model_name]

    # The final preprocessor was fitted once on X_fit_raw and is not refitted.
    assert set(final_parameters['fit_indices']) == set(X_fit_raw.index)
    assert set(final_parameters['fit_indices']).isdisjoint(reserved_indices)
    X_validation = apply_preprocessing(X_validation_raw, final_parameters)
    assert list(X_fit.columns) == list(X_validation.columns) == final_parameters['retained_columns']
    final_scaler, X_fit_model, X_validation_model = fit_optional_scaler(
        X_fit, X_validation, selected_spec['needs_scaling']
    )
    final_model = instantiate_model(selected_spec, selected_params)
    final_model.fit(X_fit_model, y_fit)
    assert int(final_model.n_features_in_) == X_fit.shape[1]

    validation_scores = get_model_scores(final_model, X_validation_model)
    selected_threshold, threshold_table = select_threshold(
        y_validation, validation_scores, selected_spec['default_threshold']
    )
    threshold_table.insert(0, 'model', selected_model_name)
    threshold_table.insert(1, 'best_params', format_best_params(selected_params))

    # Test is transformed and scored only after every selection is fixed.
    X_test = apply_preprocessing(X_test_raw, final_parameters)
    assert list(X_fit.columns) == list(X_validation.columns) == list(X_test.columns)
    assert X_fit.shape[1] == X_validation.shape[1] == X_test.shape[1]
    X_test_model = final_scaler.transform(X_test) if final_scaler is not None else X_test.to_numpy()
    test_evaluation_count = 0
    final_test_scores = get_model_scores(final_model, X_test_model)
    test_evaluation_count += 1
    final_row = evaluate_predictions(selected_model_name, y_test, final_test_scores, selected_threshold, selected_params)
    final_metrics = pd.DataFrame([final_row])
    confusion = final_metrics[['model', 'selected_threshold', 'tn', 'fp', 'fn', 'tp']].copy()
    assert test_evaluation_count == 1
    assert len(final_metrics) == 1 and final_metrics.iloc[0]['model'] == selected_model_name
    assert int(confusion[['tn', 'fp', 'fn', 'tp']].sum(axis=1).iloc[0]) == len(y_test)

    protocol = pd.DataFrame([
        ['outer split', 'raw admission-safe data', 'fit/validation/test', 'all partitions are disjoint before learned preprocessing'],
        ['manual CV preprocessing', 'each CV subfold train only', '5 deterministic stratified folds', 'missingness, imputation, clipping, and scaling exclude each subfold validation'],
        ['model selection', 'fit partition folds only', 'all model/hyperparameter combinations', 'highest mean balanced accuracy; lower standard deviation and parameter signature break ties'],
        ['final preprocessing and fit', 'fit partition only', selected_model_name, 'one preprocessor, scaler, and model; validation is not fitted'],
        ['threshold selection', 'validation partition only', selected_model_name, 'same fitted model and feature schema; no refit follows'],
        ['final evaluation', 'test partition exactly once', selected_model_name, 'same fitted model, scaler, preprocessor, schema, and locked threshold'],
    ], columns=['stage', 'data_used', 'scope', 'rule'])

    cv_summary.to_csv(CV_RESULTS_SUMMARY_TABLE, index=False)
    cv_fold_results.to_csv(CV_FOLD_RESULTS_TABLE, index=False)
    cv_fold_audit.to_csv(CV_FOLD_AUDIT_TABLE, index=False)
    baseline_vs_weighted.to_csv(BASELINE_VS_WEIGHTED_TABLE, index=False)
    threshold_table.to_csv(THRESHOLD_SELECTION_TABLE, index=False)
    final_metrics.to_csv(FINAL_MODEL_METRICS_TABLE, index=False)
    confusion.to_csv(FINAL_CONFUSION_MATRICES_TABLE, index=False)
    protocol.to_csv(EVALUATION_PROTOCOL_TABLE, index=False)
    plot_threshold_tradeoffs(threshold_table, selected_model_name)
    curve = compute_roc_curve_points(y_test, final_test_scores)
    plot_roc_curve(selected_model_name, final_row['roc_auc'], curve)
    plot_tree_pruning_curve(cv_fold_results.loc[cv_fold_results['model'] == 'DecisionTreeClassifier'])

    summary = {
        'fit_shape': X_fit.shape,
        'validation_shape': X_validation.shape,
        'held_out_test_shape': X_test.shape,
        'identical_final_feature_count': X_fit.shape[1],
        'preselected_model': selected_model_name,
        'selection_cv_balanced_accuracy': float(cv_summary.iloc[0]['best_cv_balanced_accuracy']),
        'selected_threshold': selected_threshold,
        'preprocessor_fit_rows': len(final_parameters['fit_indices']),
        'scaler_fit_rows': int(final_scaler.n_samples_seen_) if final_scaler is not None else 0,
        'validation_used_for_fit': False,
        'test_evaluation_count': test_evaluation_count,
        'test_evaluated_models': [selected_model_name],
    }
    return cv_summary, cv_fold_results, cv_fold_audit, baseline_vs_weighted, threshold_table, final_metrics, confusion, protocol, summary


## 1. Dataset description

The UCI myocardial infarction complications dataset contains 1,700 records and 124 columns. This analysis predicts relapse of myocardial infarction (`REC_IM`, column 122) from information available at hospital admission. The binary target is clinically relevant and markedly imbalanced, which motivates balanced accuracy as the primary selection metric.

The admission window includes input columns 2–112 except columns 93, 94, 95, and 100–105. The identifier, all non-target outcomes, and these nine post-admission variables are excluded exactly as specified by the dataset documentation.


In [6]:
raw_df = load_raw_dataset()
model_df, exclusion_audit, feature_columns = build_modeling_dataframe(raw_df)
X_raw = model_df.drop(columns=[TARGET_COLUMN]).copy()
y = model_df[TARGET_COLUMN].copy()

# The outer split occurs before missingness selection, imputation, clipping, scaling, or model selection.
X_development_raw, X_test_raw, y_development, y_test = train_test_split(
    X_raw, y, train_size=0.8, random_state=RANDOM_STATE, stratify=y
)
X_fit_raw, X_validation_raw, y_fit, y_validation = train_test_split(
    X_development_raw, y_development, train_size=0.75, random_state=RANDOM_STATE, stratify=y_development
)

fit_indices = set(X_fit_raw.index)
validation_indices = set(X_validation_raw.index)
test_indices = set(X_test_raw.index)
assert fit_indices.isdisjoint(validation_indices)
assert fit_indices.isdisjoint(test_indices)
assert validation_indices.isdisjoint(test_indices)
assert fit_indices | validation_indices == set(X_development_raw.index)

exclusion_audit.to_csv(EXCLUSION_AUDIT_TABLE, index=False)
print('Raw shape:', raw_df.shape)
print('Admission-safe predictors:', len(feature_columns))
print('Fit/validation/test rows:', len(X_fit_raw), len(X_validation_raw), len(X_test_raw))
print('Target counts:')
print(y.value_counts().sort_index())
print('Excluded admission-time columns:')
print(exclusion_audit.loc[exclusion_audit['column_name'].isin(ADMISSION_FORBIDDEN_COLUMNS), ['column_number', 'column_name']].to_string(index=False))


Raw shape: (1700, 124)
Admission-safe predictors: 102
Fit/validation/test rows: 1020 340 340
Target counts:
REC_IM
0    1541
1     159
Name: count, dtype: int64
Excluded admission-time columns:
 column_number column_name
            93    R_AB_1_n
            94    R_AB_2_n
            95    R_AB_3_n
           100    NA_R_1_n
           101    NA_R_2_n
           102    NA_R_3_n
           103  NOT_NA_1_n
           104  NOT_NA_2_n
           105  NOT_NA_3_n


In [7]:
# This is the single preprocessor used by the final model on fit, validation, and test.
final_parameters, missingness_summary, feature_handling, clip_summary = fit_preprocessing(
    X_fit_raw, 'final fit partition'
)
assert set(final_parameters['fit_indices']) == fit_indices
assert set(final_parameters['fit_indices']).isdisjoint(validation_indices | test_indices)
X_fit = apply_preprocessing(X_fit_raw, final_parameters)

feature_handling.to_csv(FEATURE_HANDLING_TABLE, index=False)
clip_summary.to_csv(SIGMA_CLIPPING_TABLE, index=False)

# Target-linked descriptive analyses remain inside the fit partition.
X_analysis = X_fit.copy()
y_analysis = y_fit.copy()
print('Final feature count learned from fit only:', X_fit.shape[1])
print('Fit missing values after transformation:', int(X_fit.isna().sum().sum()))
print('Dropped by the fit-only missingness rule:', final_parameters['dropped_columns'])
print('Final preprocessor fit scope:', final_parameters['stage'])


Final feature count learned from fit only: 97
Fit missing values after transformation: 0
Dropped by the fit-only missingness rule: ['KFK_BLOOD', 'IBS_NASL', 'D_AD_KBRIG', 'S_AD_KBRIG', 'NOT_NA_KB']
Final preprocessor fit scope: final fit partition


## 2. Data cleaning

Preprocessing is an explicitly fitted transformation: predictors above 40% missingness are removed, low-cardinality variables receive mode imputation, other variables receive median imputation, and continuous values are clipped to training-derived mean ± 3 standard deviations. During manual cross-validation, every fold learns these decisions from its own subfold training rows and applies them unchanged to its subfold validation rows. After selection, one final preprocessor is fitted on `X_fit_raw` only and is reused without refitting for validation and test.


In [8]:
print(f'Continuous predictors clipped with fit-derived {SIGMA_CLIPPING_N}-sigma bounds:', len(clip_summary))
print('Fit values outside the learned bounds:', int(clip_summary['n_clipped_training'].sum()))
print(clip_summary.sort_values('pct_clipped_training', ascending=False).head(10).round(4).to_string(index=False))
print('Saved fit-only preprocessing evidence to:')
print('-', FEATURE_HANDLING_TABLE)
print('-', SIGMA_CLIPPING_TABLE)


Continuous predictors clipped with fit-derived 3-sigma bounds: 9
Fit values outside the learned bounds: 135
  feature     mean     std  lower_bound  upper_bound  n_clipped_below_training  n_clipped_above_training  n_clipped_training  pct_clipped_training           fitted_on
ALT_BLOOD   0.4569  0.3478      -0.5864       1.5001                         0                        29                  29                2.8431 final fit partition
AST_BLOOD   0.2507  0.1794      -0.2877       0.7890                         0                        18                  18                1.7647 final fit partition
D_AD_ORIT  82.5000 16.2876      33.6373     131.3627                        13                         4                  17                1.6667 final fit partition
S_AD_ORIT 133.9216 27.7892      50.5539     217.2892                         8                         8                  16                1.5686 final fit partition
  L_BLOOD   8.7567  3.3116      -1.1783      18.6916     

## 3. Exploratory analysis

The descriptive analysis is restricted to the fit partition: class balance, missingness, selected admission-variable distributions, and simple target correlations cannot expose validation or test labels. PCA is not used in supervised preprocessing or model fitting; it appears only as a two-dimensional display of K-means clusters.


In [9]:
top_correlations = compute_top_target_correlations(X_analysis, y_analysis, top_n=10)
plot_and_export_eda(X_analysis, y_analysis, missingness_summary, top_correlations)
print('Class balance:')
print(pd.read_csv(CLASS_BALANCE_TABLE).to_string(index=False))
print('Top simple correlations:')
print(top_correlations.round(4).to_string(index=False))


Class balance:
 REC_IM  count    share
      0    925 0.906863
      1     95 0.093137
Top simple correlations:
     feature  correlation_with_target  absolute_correlation
  zab_leg_02                   0.1184                0.1184
         AGE                   0.1057                0.1057
   STENOK_AN                   0.0979                0.0979
       np_01                   0.0978                0.0978
   FK_STENOK                   0.0891                0.0891
    IBS_POST                   0.0850                0.0850
     L_BLOOD                   0.0761                0.0761
      NITR_S                   0.0748                0.0748
n_p_ecg_p_07                   0.0689                0.0689
     GT_POST                   0.0636                0.0636


### 3.1 Unsupervised exploration: K-means clustering

K-means provides a descriptive phenotyping analysis of the admission-time feature space. The silhouette score selects the number of clusters. PCA is then fitted solely to project the scaled clustering matrix into two dimensions for visualisation; principal components are never supplied to any supervised classifier.


In [10]:
best_k, cluster_labels, clustering_results, X_scaled_for_clustering = run_kmeans_clustering(X_analysis)
clustering_results.to_csv(CLUSTERING_RESULTS_TABLE, index=False)
plot_clustering(X_scaled_for_clustering, cluster_labels, clustering_results, y_analysis, best_k)

print(f'Best K selected by silhouette score: {best_k}')
print(clustering_results.round({'silhouette_score': 4}).to_string(index=False))
print('Cluster relapse rates:')
for cluster_id in sorted(set(cluster_labels)):
    mask = cluster_labels == cluster_id
    print(f"Cluster {cluster_id}: n={mask.sum()}, REC_IM=1 rate={y_analysis.to_numpy()[mask].mean() * 100:.1f}%")


Best K selected by silhouette score: 2
 k  silhouette_score                                             cluster_sizes
 2            0.2599                                          {0: 104, 1: 916}
 3            0.0261                                   {0: 415, 1: 59, 2: 546}
 4            0.0237                            {0: 45, 1: 485, 2: 58, 3: 432}
 5            0.0046                   {0: 353, 1: 187, 2: 148, 3: 258, 4: 74}
 6            0.0123              {0: 416, 1: 214, 2: 319, 3: 59, 4: 2, 5: 10}
 7            0.0201        {0: 309, 1: 2, 2: 59, 3: 208, 4: 422, 5: 1, 6: 19}
 8            0.0230 {0: 2, 1: 406, 2: 300, 3: 45, 4: 16, 5: 42, 6: 208, 7: 1}
Cluster relapse rates:
Cluster 0: n=104, REC_IM=1 rate=5.8%
Cluster 1: n=916, REC_IM=1 rate=9.7%


## 4. Main analysis: objective and methods adopted

Logistic regression, an entropy decision tree, and SVC are compared by deterministic manual five-fold stratified cross-validation on the raw fit partition. For every model/hyperparameter combination and fold, missingness selection, imputation, clipping, and scaling are fitted exclusively on that fold's training rows; the fold validation rows are transformed without refitting. Mean balanced accuracy is the primary criterion, followed by lower fold standard deviation and the parameter signature as deterministic tie-breakers.

After CV fixes the family and hyperparameters, one preprocessor, optional scaler, and estimator are fitted on `X_fit_raw` only. The same fitted objects and identical feature schema produce validation scores for threshold selection and, after the threshold is locked, the single test evaluation. Validation is reserved for threshold selection and is not included in model fitting; no refit occurs after threshold selection.


In [11]:
(
    cv_summary_table,
    cv_fold_results_table,
    cv_fold_audit_table,
    baseline_vs_weighted_table,
    validation_threshold_table,
    final_metrics_table,
    confusion_table,
    evaluation_protocol,
    selection_summary,
) = train_select_and_test(
    X_fit_raw,
    X_fit,
    final_parameters,
    X_validation_raw,
    X_test_raw,
    y_fit,
    y_validation,
    y_test,
    validation_indices | test_indices,
)

print('Selection and test protocol:')
print(evaluation_protocol.to_string(index=False))
print('Manual fold-local CV summary:')
print(cv_summary_table[['cv_rank', 'model', 'best_cv_balanced_accuracy', 'cv_score_std', 'selected_class_weight', 'best_params']].round(4).to_string(index=False))
print('Selected validation threshold:')
print(validation_threshold_table.loc[validation_threshold_table['selected'], ['model', 'threshold', 'balanced_accuracy', 'recall', 'precision', 'f1']].round(4).to_string(index=False))
print('Single final test result:')
print(final_metrics_table.round(4).to_string(index=False))
print('Runtime isolation evidence:', selection_summary)


Selection and test protocol:
                      stage                   data_used                                 scope                                                                                        rule
                outer split     raw admission-safe data                   fit/validation/test                                    all partitions are disjoint before learned preprocessing
    manual CV preprocessing  each CV subfold train only      5 deterministic stratified folds              missingness, imputation, clipping, and scaling exclude each subfold validation
            model selection    fit partition folds only all model/hyperparameter combinations highest mean balanced accuracy; lower standard deviation and parameter signature break ties
final preprocessing and fit          fit partition only                                   SVC                               one preprocessor, scaler, and model; validation is not fitted
        threshold selection   validation 

## 5. preview/summary of results

The CV-preselected SVC is the only model evaluated on the held-out test partition. The summary below reports its validation-selected threshold and the single final test result exported to the evidence tables.


In [12]:
final_result = final_metrics_table.iloc[0]
selected_threshold_row = validation_threshold_table.loc[validation_threshold_table['selected']].iloc[0]
print('Preselected final model:', final_result['model'])
print(f"CV balanced accuracy used for family selection: {selection_summary['selection_cv_balanced_accuracy']:.4f}")
print(f"Validation-selected threshold: {final_result['selected_threshold']:.4f}")
print(
    f"Single held-out test result — balanced accuracy: {final_result['balanced_accuracy']:.4f}; "
    f"recall: {final_result['recall']:.4f}; precision: {final_result['precision']:.4f}; "
    f"F1: {final_result['f1']:.4f}; ROC-AUC: {final_result['roc_auc']:.4f}."
)


Preselected final model: SVC
CV balanced accuracy used for family selection: 0.6253
Validation-selected threshold: 0.5056
Single held-out test result — balanced accuracy: 0.5244; recall: 0.2500; precision: 0.1143; F1: 0.1569; ROC-AUC: 0.5597.


## 6. Detailed results

The exported evidence separates fold-local model selection, validation-only threshold selection, and the single final test evaluation. `cv_fold_preprocessing_audit.csv` records all model/parameter/fold boundaries, while `evaluation_protocol.csv` records the outer boundaries. The threshold figure contains only the CV-selected SVC because no other family is evaluated on validation, and the ROC figure contains only that same fitted SVC because competing families are never evaluated on test.


In [13]:
print('Cross-validation family ranking:')
print(cv_summary_table.round(4).to_string(index=False))
print('Class-weight comparison within CV:')
print(baseline_vs_weighted_table.round(4).to_string(index=False))
print('Validation threshold selected for the CV winner:')
print(validation_threshold_table.loc[validation_threshold_table['selected']].round(4).to_string(index=False))
print('Final test confusion counts:')
print(confusion_table.to_string(index=False))


Cross-validation family ranking:
 cv_rank                  model                                                                                       selection_criterion                                                               best_params  best_cv_balanced_accuracy  cv_score_std  fold_1_balanced_accuracy  fold_2_balanced_accuracy  fold_3_balanced_accuracy  fold_4_balanced_accuracy  fold_5_balanced_accuracy  param_C param_class_weight  param_max_depth  param_min_samples_leaf  param_ccp_alpha param_kernel  needs_scaling selected_class_weight  default_threshold
       1                    SVC highest mean fold-local CV balanced accuracy; lower standard deviation and parameter signature break ties                               C=0.1; class_weight=balanced; kernel=linear                     0.6253        0.0437                    0.6152                    0.5808                    0.5956                    0.7063                    0.6287      0.1           balanced              NaN  

## 7. Conclusions

The admission-time information boundary is preserved by excluding the identifier, all non-target outcomes, and columns 93–95 and 100–105. Every CV fold learns preprocessing and scaling only from its subfold training rows. Model-family and hyperparameter selection use fit-partition folds; threshold selection uses reserved validation data; and the same fitted preprocessor, scaler, feature schema, and estimator are evaluated once on test. PCA serves only as a two-dimensional cluster visualisation and is not part of supervised modeling.


In [ ]:
result = final_metrics_table.iloc[0]
print(f"The CV criterion selected {result['model']} before validation and test evaluation.")
print(
    f"At the validation-selected threshold of {result['selected_threshold']:.4f}, the single test evaluation produced "
    f"balanced accuracy {result['balanced_accuracy']:.4f}, recall {result['recall']:.4f}, "
    f"precision {result['precision']:.4f}, F1 {result['f1']:.4f}, and ROC-AUC {result['roc_auc']:.4f}."
)
print(f"Confusion counts were TN={int(result['tn'])}, FP={int(result['fp'])}, FN={int(result['fn'])}, and TP={int(result['tp'])}.")


The CV criterion selected SVC before validation and test evaluation.
At the validation-selected threshold of 0.5056, the single test evaluation produced balanced accuracy 0.5244, recall 0.2500, precision 0.1143, F1 0.1569, and ROC-AUC 0.5597.
Confusion counts were TN=246, FP=62, FN=24, and TP=8.
These results indicate limited admission-time discrimination and should not be interpreted as clinical validation.
